# Ollama Server

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tgz
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess, time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # give it time to start

In [5]:
import requests
requests.get("http://localhost:11434").status_code

200

In [6]:
!ollama pull gpt-oss:20b

In [7]:
!pip install ollama

In [8]:
from ollama import chat

response = chat(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "تعرف تتكلم عربي مصري؟"}
    ]
)

print(response["message"]["content"])

ResponseError: model requires more system memory (13.1 GiB) than is available (12.1 GiB) (status code: 500)

In [9]:
!pip install langchain langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 whi

In [10]:
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

chat = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434"
)

messages = [
    SystemMessage(content="You are a helpful AI assistant."),
    HumanMessage(content="Explain transformers briefly")
]

response = chat.invoke(messages)
print(response.content)

/tmp/ipython-input-505788549.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  chat = ChatOllama(


**Transformers** are a family of neural‑network architectures that have become the foundation of modern natural‑language processing (NLP) and many other sequence‑processing tasks. They were introduced in the 2017 paper *“Attention Is All You Need”* by Vaswani et al. and are notable for dispensing with recurrence and convolution in favor of a pure‑attention mechanism.

---

### Core Ideas

| Component | What it does | Why it matters |
|-----------|--------------|----------------|
| **Self‑attention (Scaled Dot‑Product Attention)** | For each token in a sequence, it learns *how much* to attend to every other token, producing a weighted sum of their representations. | Allows the model to capture long‑range dependencies directly and in parallel. |
| **Multi‑head attention** | Runs several self‑attention “heads” in parallel, each learning different aspects (e.g., syntactic vs. semantic relationships). | Enriches the representation with diverse relational patterns. |
| **Positional encoding*

In [11]:
for chunk in chat.stream("Explain attention"):
    print(chunk.content, end="", flush=True)

## Attention – the “focus” mechanism that powers modern deep learning

Attention lets a neural network decide **which parts of an input are most relevant to a particular task**.  
It was invented for machine translation (Bahdanau et al., 2014), then re‑invented in the Transformer
(`Attention Is All You Need`, 2017), and today it appears in NLP, computer vision, speech, and more.

Below is a “cook‑book” style walk‑through:

---

### 1.  Intuition – What is attention?

Imagine you’re translating a sentence “The **cat** sat on the **mat**.”  
When the model is predicting the word *mat*, it should **pay more attention to the word “cat”** because
they belong to the same syntactic or semantic cluster (both are concrete nouns).  
Conversely, when it predicts *sat*, it should focus on *cat* and *mat*.

Attention lets the network automatically learn *which tokens influence which outputs*,
instead of forcing all tokens to contribute equally.

---

### 2.  The building blocks

| Symbol | Meaning 

# RAG LangChain

In [17]:
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [18]:
chat = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0.2
)


In [19]:
docs = [
    Document(page_content="RAG means Retrieval-Augmented Generation."),
    Document(page_content="FAISS is used for similarity search.")
]

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever()

In [20]:
prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the context below.

Context:
{context}

Question:
{question}
""")

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | chat
)


In [21]:
response = rag_chain.invoke("What is RAG?")
print(response.content)

RAG means Retrieval-Augmented Generation.


# Streaming the output

In [ ]:
import sys
from ollama import generate

for chunk in generate(model="gpt-oss:20b", prompt="Explain transformers in Egyptian Arabic Dileact", stream=True):
    sys.stdout.write(chunk["response"])
    sys.stdout.flush()

## إيه هو الـ Transformers؟

الـ Transformers هو نوع من نماذج الذكاء الاصطناعي (الـ neural networks) اللي اتعرفوا عليها في عام 2017، وأصبحوا السبب وراء معظم التقدم اللي شفناه في فهم اللغة، الترجمة، الكتابة، وحتى ألعاب الفيديو!  
فيه شوية تفاصيل مهمة، بس هنحاول نوضحها بأبسط طريقة.

---

### ١. إيه اللي يميز الـ Transformers؟

قبل الـ Transformers كان معظم النماذج في المجال، زي الـ RNNs و LSTMs، بتحاول تقرأ الجمل خطوة بخطوة (من اليسار للنّهاية أو العكس). ده خلى التدريب صعب ومرهق، وكمان ما كانش ممكن إن النموذج يشتغل بسرعة على الجمل الطويلة.

الـ Transformers جاب فكرة جديدة: **الـ “attention” (الانتباه)**. بدل ما يقرأ الجمل خطوة بخطوة، بيشوف كل كلمة في الجملة وتعرف على العلاقة بينها وبين الكلمات التانية في نفس الجملة في وقت واحد. فـ يقدر يركز على أهم الكلمات اللي بتأثر على معنى الجملة كلها.

---

### ٢. الخطوات الأساسية في Transformer

| الخطوة | ماذا تفعل | مثال بسيط |
|--------|-----------|-----------|
| **الـ Embedding** | تحول الكلمات إلى أرقام (vectors) | “القط” -> [0.45, 1.2, …] |
| 

In [23]:
from langchain_community.chat_models import ChatOllama

chat = ChatOllama(
    model="gpt-oss:20b",
    base_url="http://localhost:11434",
    temperature=0.2,
    streaming=True  # IMPORTANT
)

# Stream messages
for token in chat.stream("Explain attention in simple terms"):
    print(token.content, end="", flush=True)


### Attention – the “focus” trick that lets models pick out what matters

Think of a human reading a sentence.  
You don’t read every word with the same intensity.  
When you see the word **“cat”** you automatically look back at the word **“love”** to understand the meaning.  
You’re *attending* to the parts of the sentence that help you answer the question: “What is the sentence about?”

That’s exactly what the *attention mechanism* does in a neural network, but in a way that can be computed quickly and automatically.

---

## 1. The core idea

For every piece of input (a word, a pixel, a sound sample, …) the model decides **how much it should “pay attention” to every other piece**.  
The result is a weighted sum:  
- *High weight* → the model thinks this part is very important.  
- *Low weight* → the model thinks this part is less relevant.

The weights are learned from data, so the model figures out the right pattern of focus on its own.

---

## 2. How it works in a nutshell

1. **